## Apartado 5. Resumen automático abstractivo



### **5.1 Utilizar un modelo pre-entrenado para crear y guardar el resumen de un texto en un archivo JSON**

En este apartado 5 vamos a trabajar la tarea de **resumen automático abstractivo** que consiste en generar de forma automática un texto que recoja y resuma las ideas principales del contenido del propio texto mediante nuevas frases que reflejen el contenido y el sentido del texto original.

Para ello, vamos a utilizar un modelo ya preentrenado de Hugging Face llamado **mT5_multilingual_XLSum**, que está diseñado para tareas de resumen. En nuestro caso, para cada hilo de Reddit vamos a utilizar este modelo utilizando el texto de la descripción y del título de cada submission y el cuerpo de cada comentario, permitiendo al modelo coger buen contexto y generar un resumen más completo.

Una vez que se haya generado el resumen, lo añadiremos como un nuevo campo dentro de los parámetros de las submissions de cada JSON de cada subreddit, manteniendo también toda la información original.

In [ ]:
!pip install -U transformers huggingface_hub bitsandbytes

Antes de nada, definimos el modelo extraído de Hugging Face y los cargamos juntos con su tokenizador, para poder procesar los textos y generar los resúmenes automáticos.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import json
import torch

# Definimos el modelo que vamos a usar
model_path = "csebuetnlp/mT5_multilingual_XLSum"

# Cargamos el tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Cargamos el modelo
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


A continuación, recorremos cada uno de los subreddits seleccionados y cargamos sus archivos JSON correspondientes. Para cada submission, construimos el texto de entrada que recibirá el modelo uniendo el título del hilo, la descripción principal y el contenido de todos sus comentarios, de forma que el resumen se genere teniendo en cuenta el contexto completo de la publicación.

Posteriormente, ese texto se tokeniza y se introduce en el modelo mT5, que genera un resumen abstractivo del mismo. El resultado obtenido se almacena en un nuevo campo llamado resumen_abstractivo_mT5 dentro de cada submission.

Por último, una vez procesado todo el subreddit, se crea un nuevo archivo JSON que conserva la información original e incorpora los resúmenes generados automáticamente para cada hilo.

In [ ]:
# Definimos los subreddits
subreddits = ["onepiece", "soccer", "gaming", "movies", "leagueoflegends", "drawing"]

# Recorremos los subreddits
for subreddit in subreddits:

    filename = subreddit + "_final.json"

    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"Procesando {filename}...")

    # Recorremos las submissions
    for submission in data["submissions"]:

        # Cogemos el título de la submission
        titulo = str(submission.get("title", "")).strip()

        # Cogemos la descripción de la submission
        selftext = str(submission.get("selftext", "")).strip()

        # Creamos una lista para almenecesar los textos de los cuerpos de los comentarios de cada submission
        comentarios = []

        # Cogemos el cuerpo de cada comentario
        for comment in submission["comments"]:
            cuerpo = str(comment.get("body", "")).strip()
            comentarios.append(cuerpo)

        # Unimos título + descripción + comentarios
        texto = (titulo + " " + selftext + " " + " ".join(comentarios)).strip()

        # Si no hay texto, dejamos resumen vacío
        if texto == "":
            submission["resumen_abstractivo_mT5"] = ""
            continue

        # Generamos el resumen de la descripción
        try:
            # Tokenizamos el texto de entrada
            inputs = tokenizer(
                texto,
                max_length=512,
                truncation=True,
                return_tensors="pt"
            ).to(model.device)

            # Generamos la respuesta del modelo
            outputs = model.generate(
                **inputs,
                max_length=60,
                min_length=15,
                num_beams=4,
                early_stopping=True
            )

            # Decodificamos el resumen generado
            response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        except:
            response = ""

        # Guardamos el resumen en una variable del JSON
        submission["resumen_abstractivo_mT5"] = response

    # Creamos un nuevo archivo JSON para añadir la variable
    nuevo_nombre = subreddit + "_resumen_abstractivo.json"

    with open(nuevo_nombre, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

Procesando onepiece_final.json...
Procesando soccer_final.json...
Procesando gaming_final.json...
Procesando movies_final.json...
Procesando leagueoflegends_final.json...
Procesando drawing_final.json...


Una vez ejecutado el código anterior, hemos generado los archivos JSON que incluyen para cada submission, una nueva variable que contiene el resumen abstractivo obtenido con el modelo mT5. A continuación, en el siguiente apartado, vamos a generar los mismos resuménes de los submissions pero utilizando el modelo SLM Gemma. Una vez terminado dicho apartado y generados los resumenes abstractivos con ambos modelos, procederemos a analizar cada resumen y a comparar los resultados entre modelos.

<!--  -->

#### **5.2 Utilizar SLMs (Small Large Models) para crear y guardar el resumen de un texto en un archivo JSON**

Zero-shot learning es una técnica en la que un modelo resuelve una tarea sin haber visto ejemplos específicos de cómo resolverla. Simplemente recibe una lista de instrucciones en el promt y el modelo intenta generar una respuesta basándose en el conocimiento adquirido durante su preentrenamiento.

En este apartado vamos a utilizar SLMs (Small Language Models) para generar resúmenes automáticos de los hilos de cada subreddit y los almacenaremos posteriormente en una variable dentro del archivo JSON. A diferencia del apartado anterior, donde se empleó un modelo específicamente entrenado para resumen, en este caso trabajaremos con **Gemma**.

Para ello, aplicaremos una estrategia de **Zero-Shot Learning**, que consiste en pedir directamente al modelo que realice una tarea sin que haya visto como hacerlo mediante ejemplos previos. Es decir, el modelo recibirá únicamente un prompt con una instrucción clara indicando que debe resumir el contenido proporcionado.

El texto que se resumirá en cada hilo estará formado por el título y la descripción de cada submission junto con el cuerpo de los comentarios del hilo, con el objetivo de aportar al modelo el máximo contexto posible. Cuando se haya creado el resumen, lo añadiremos como una nueva variable dentro de cada submission y crearemos un archivo JSON nuevo cuando se recorra todo un subreddit.

Antes de nada, para poder trabajar con el modelo de Hugging Face debemos pedir permisos desde la propia página y cargar el modelo mediante un token que hemos creado. Para utilizar este token he creado un 'secret' de forma que utilizo el token sin mostrarlo directamente en la práctica (esto es solo por seguridad).

In [ ]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("token_hugginface")
login(token)

In [ ]:
!pip install -U bitsandbytes accelerate transformers

Ahora definimos el modelo que vamos a utilizar en este apartado, en este caso Gemma-3-1b-it.

Si existe una GPU disponible en el entorno de ejecución, aplicamos una cuantización de 4 bits, una técnica que reduce el tamaño del modelo y permite ejecutarlo de forma más eficiente, consumiendo menos recursos.

Después, cargamos tanto el tokenizador y el modelo.

In [ ]:
# Importamos las librerias necesarias
import json
import torch
from transformers import pipeline, AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM

# Definimos el modelo que vamos a usar
model_path = "google/gemma-3-1b-it"

if torch.cuda.is_available():
  # Configuración la cuantización de 4-bit
  quantization_config = BitsAndBytesConfig(load_in_4bit=True)
else:
  quantization_config = None

# Cargamos el tokenizador
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Cargamos el modelo
# Estamos cuantizando el modelo a 4-bits para que ocupe menos espacio en GPU
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=quantization_config,
    device_map="auto"
).eval()

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

A continuación, recorremos cada uno de los subreddits seleccionados y cargamos sus archivos JSON correspondientes. Para cada submission, construimos el texto de entrada que recibirá el modelo uniendo el título del hilo, la descripción principal y el contenido de todos sus comentarios, de forma que el resumen se genere teniendo en cuenta el contexto completo de la publicación.

Una vez tenemos el texto, lo introducimos en el modelo Gemma mediante un enfoque de Zero-Shot Learning, utilizando un prompt que indica explícitamente que debe generar un resumen en una sola oración, sin añadir información adicional.

Después, aplicamos un template de chat al mensaje de entrada utilizando el tokenizador. Esto formatea los mensajes según el formato esperado por el modelo, añadiendo un prompt y posteriormente generamos la respuesta del modelo.

Una vez obtenemos el resumen respuesta del modelo gemma, lo añadimos como una variable en los campos del submission utilizando los mismos JSON que creamos en el apartado 5.1, de forma que podamos comparar la eficiencia de ambos resumenes más fácilmente.

In [22]:
# Definimos los subreddits
subreddits = ["onepiece", "soccer", "gaming", "movies", "leagueoflegends", "drawing"]

# Recorremos los subreddits
for subreddit in subreddits:

    filename = subreddit + "_resumen_abstractivo.json"

    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"Procesando {filename}...")

    # Recorremos las submissions
    for submission in data["submissions"]:

        # Cogemos el título del hilo
        titulo = str(submission.get("title", "")).strip()

        # Cogemos la descripción del hilo
        selftext = str(submission.get("selftext", "")).strip()

        # Creamos una lista para almenecesar los textos de los cuerpos de los comentarios de cada submission
        comentarios = []

        # Cogemos el cuerpo de cada comentario
        for comment in submission["comments"]:
            cuerpo = str(comment.get("body", "")).strip()
            comentarios.append(cuerpo)

        # Unimos título + descripción + comentarios
        texto = (titulo + " " + selftext + " " + " ".join(comentarios)).strip()

        # Si no hay texto, dejamos resumen vacío
        if texto == "":
            submission["resumen_abstractivo_gemma"] = ""
            continue

        # Creamos el prompt zero-shot donde pedimos que haga un resumen breve y detallado
        prompt = f"""
                     Eres un modelo cuya tarea es resumir el texto recibido en UNA sola oración.

                    - Escribe solo una oración.
                    - No uses saltos de línea.
                    - No inventes información.

                    Texto: {texto}

                    Resumen:
                  """

        try:
            # Estructuramos los mensajes de entrada en el formato requerido por Gemma
            messages = [
                {
                    "role": "user",
                    "content": prompt + " " + texto,
                },
            ]

            # Aplicamos un template de chat al mensaje de entrada utilizando el tokenizador
            inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_tensors="pt",
                return_dict=True
            ).to(model.device)

            # Generamos la respuesta del modelo
            outputs = model.generate(
                **inputs,
                max_new_tokens=50,
                temperature=0.9,
                top_k=35,
                top_p=0.95,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

            # Decodificamos únicamente la parte generada, sin incluir el prompt de entrada (este codigo está en hugging face)
            response = tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[-1]:],
                skip_special_tokens=True
            ).strip()

        except Exception as e:
            print("Error:", e)
            response = ""

        # Guardamos el resumen en una variable de submission en el JSON
        submission["resumen_abstractivo_gemma"] = response

    # Creamos un nuevo archivo JSON con los resumenes
    nombre_json = subreddit + "_resumenes_abstractivos.json"

    with open(nombre_json, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"Guardado {nombre_json}")

Procesando onepiece_resumen_abstractivo.json...
Guardado onepiece_resumenes_abstractivos.json
Procesando soccer_resumen_abstractivo.json...
Guardado soccer_resumenes_abstractivos.json
Procesando gaming_resumen_abstractivo.json...
Guardado gaming_resumenes_abstractivos.json
Procesando movies_resumen_abstractivo.json...
Guardado movies_resumenes_abstractivos.json
Procesando leagueoflegends_resumen_abstractivo.json...
Guardado leagueoflegends_resumenes_abstractivos.json
Procesando drawing_resumen_abstractivo.json...
Guardado drawing_resumenes_abstractivos.json


En este punto, ya tenemos los ficheros JSON finales, que incluyen para cada submission tanto el resumen generado con el modelo mT5 como el obtenido con Gemma. A continuación, vamos a analizar los resultados de cada modelo y a realizar una comparación de ambos modelos, observando la calidad de los resúmenes generados.

<!--  -->

### **5.3 Análisis y comparación de los resumenes abstractivos generados con ambos modelos**

En este punto, ya tenemos los ficheros JSON finales, que incluyen para cada submission tanto el resumen generado con el modelo mT5 como el obtenido con Gemma. A continuación, vamos a analizar los resultados de cada modelo y a realizar una comparación de los resúmenes generados utilizando algunos ejemplos de algunos de los resúmenes abstractivos de ambos modelos para varios hilos.

<!--  -->

#### **PRIMER HILO:**

**Descripción del hilo**

El primer hilo que vamos a analizar lo hemos extraído del JSON "onepiece_resumenes_abstractivos.json". Concretamente es el segundo hilo que encontramos al abrir el fichero con id ""1p5j1d6". 

El contexto del hilo extraído trata sobre una pregunta abierta acerca del personaje 'Dragon' en One Piece, concretamente sobre el símbolo que tiene en la cara. A partir del título del propio hilo y la descripción, se plantea si se trata de una cicatriz, un tatuaje o una marca de nacimiento.

- Título: *"What is the symbol on dragon's face ?"*
- Descripción: *"Is it a scar ? A tattoo ? A birthmark ? The hell is it."*

En los comentarios del hilo, podemos ver como los usuarios aportan distintas teorías, algunas más serias y otras más humorísticas. Por ejemplo, un usuario sugiere que podría ser un tatuaje con significado cultural relacionado con tormentas, mientras que otros plantean que podría estar vinculado a los poderes obtenidos tras comerse una fruta del diablo, a su pasado en la marina o incluso a elementos mitológicos. 

En general, la discusión gira en torno a intentar deducir cuál podría el origen y significado de dicha marca. Dichos comentarios son los siguientes:

- *"It’s the markings of a monkey dragon from **japanese mythology**"*
- *"I remember seeing a video about this. It seems like a Native American Tattoo about a **storm** or something along those lines. Give me time to fine the source"*
- *"The tattoo seems to resembles eyes. It could be that he is a **recent eater of a fruit and was given the tattoo as a symbol from his power**..."*

<!--  -->

**Análisis del resumen generado por mT5**

En cuanto al resumen abstractivo generado por el modelo mT5, podemos ver que es claramente incorrecto:

- *“This is a full transcript of The Dark Knight Rises:”*

Este resumen generado no tiene ninguna relación con el contenido de los textos tratados en la submission. El modelo introduce información completamente diferente (una película de Batman que no tiene nada que ver con One Piece), lo que indica un fallo grave en la generación de textos. 

Por lo tanto, podemos dedcucir que mT5 no ha sido capaz de interpretar correctamente el contexto del texto original. El modelo parece haber perdido información relevante o haber seleccionado elementos poco representativos, lo que hace que no haya precisión ni relación del resumen generado con lo analizado. En cuanto a la calidad del resúmen, el resultado no puede ni considerarse útil, ya que no refleja las ideas principales del texto ni permite comprender su contenido de forma fiable. Por tanto, el resumen generado por mT5 no es un buen resúmen, pudiendo incluso de decir que el resumen general es **malo**.

<!--  -->

**Análisis del resumen generado por Gemma**

En cuanto al resumen abstractivo generado por Gemma, podemos ver que ha sido el siguiente:

- *"Dragon's tattoo seems to be a remnant of a Japanese legend - a potential depiction of a symbol associated with a storm or a particular type of tattoo that predates the modern era."*

En este caso, el modelo sí capta la idea general de la submission. Identifica correctamente que se está hablando de un posible tatuaje o símbolo en el personaje 'Dragon' y recoge información de algunas de las teorías planteadas en los comentarios, relacionadas con su origen cultural o simbólico y sobre la tormenta. Aunque el resumen no menciona todas las posibles hipótesis posibles (como la cicatriz o los poderes de la fruta demoníaca), sí refleja de forma coherente el tema principal del hilo. Además, tiene una redacción clara y consistente, sin fallos gramaticales y sin introducir información fuera de contexto.

<!--  -->

**Comparación entre ambos modelos**

Comparando ambos resultados, podemos apreciar una clara diferencia de calidad entre resúmenes. Mientras que mT5 genera un resumen totalmente incorrecto y distinto del contenido, Gemma produce un resumen coherente y alineado con la temática de la submission. Aunque el resumen de Gemma no es perfecto y simplifica la discusión a unos pocos comentarios, es infinitamente más útil que mt5 y más fiel al contenido original.

En conclusión, para este ejemplo, Gemma demuestra un rendimiento claramente superior al de mT5, mientras que mT5 falla completamente.

<!--  -->

#### **SEGUNDO HILO:**

**Descripción del hilo**

El segundo hilo analizado también pertenece al subreddit de One Piece y tiene el id "1ktkksn".

Su contexto se basa en la pregunta principal del título del hilo, donde se pregunta si el Luffy actual sería capaz de derrotar a Akainu.

- Título: *"Can current Luffy beat akainu"*

El hilo no tiene descripción adicional, por lo que el contexto se obtiene a partir del título y de los comentarios. En general, los usuarios discuten si el estado actual de Luffy, tras obtener el Gear 5 y convertirse en un Yonko es suficiente como para vencer a Akainu (un almirante de la Marina). Algunos comentarios dicen que sí podría derrotarlo, argumentando que Luffy ya ha vencido a enemigos poderosos (como a otro Yonko) y que actualmente es uno de los personajes más fuertes. Mientras que otros comentan algunas limitaciones como que no puede mantener el Gear 5 (una transformación) durante mucho tiempo, poniendo en duda si realmente podría. Por ejemplo:

- *“Of course he can, he’s a Yonko”*
- *"If he could hold gear 5 for longer than he currently can then sure he could beat him"*

<!--  -->

**Análisis del resumen generado por mT5**

En cuanto al resumen abstractivo generado por el modelo mT5, podemos ver que el resultado ha sido el siguiente:

- *"If you're a fan of Luffy, it's probably the most unlikely that he could beat Akainu or Blackbeard."*

Este resumen recoge más o menos el tema del hilo, ya que menciona a Luffy y Akainu, pero presenta algunos problemas. En primer lugar, nos nombra a Blackbeard (Barbanegra), que aunque aparece en uno de los comentarios, no es el foco principal de la discusión. Además, el resumen afirma que es poco probable que Luffy pueda vencer a Akainu, cuando el hilo contiene opiniones bastante divididas e incluso muchos comentarios dicen que sí que podría hacerlo.

Por tanto, mT5 no falla completamente como en el ejemplo anterior, pero sí ofrece una interpretación con algunos fallos. 

<!--  -->

**Análisis del resumen generado por Gemma**

En cuanto al resumen abstractivo generado por Gemma, podemos ver que ha sido el siguiente:

- *"Can current Luffy beat Akainu? Of course he can. He's a Yonko."*

Este resumen capta mejor la idea principal del hilo, ya que mantiene la pregunta del título sobre si Luffy puede vencer a Akainu y utiliza una de las respuestas de los comentarios donde se dice que si que podría. Además, es un resumen claro, breve y directamente relacionado con el texto original.

Sin embargo, también presenta una limitación y es que simplifica demasiado el tema. El hilo no solo contiene respuestas que dicen que sí podría vencer a Akainu, sino también comentarios que señalan que Luffy podría ganar en algunas condiciones, pero no siempre lo haría. Aun así, el resumen de Gemma es coherente y no introduce información ajena al contenido.

<!--  -->

**Comparación entre ambos modelos**


Comparando ambos resultados, podemos ver que Gemma vuelve a ofrecer un mejor resumen que mT5. mT5 identifica el tema pero genera una conclusión poco representativa utilizando datos que no se relacionan del todo. Gemma resume de forma más clara la idea principal del hilo, pero su respuesta se queda algo corta. 

Aunque Gemma no produce un resumen perfecto, frente a mT5, su resultado es más útil y fiel al tema principal. En conclusión, para este segundo ejemplo, Gemma vuelve a tener un mejor rendimiento al generar el resumen, mejor que el de mT5.

<!--  -->

#### **TERCER HILO:**

**Descripción del hilo**

El tercer hilo analizado también pertenece al subreddit de One Piece y tiene el id "1icz5u0".

Su contexto se basa en la pregunta principal del título del hilo, donde se pregunta qué isla elegiría visitar un usuario si despertase dentro del mundo de One Piece.

- Título: *"If you woke up in the world of One Piece which island will you choose to visit first?"*

El hilo no tiene descripción, por lo que el contexto se obtiene a partir del título y los comentarios. En general, los usuarios responden diciendo las distintas islas del mundo de One Piece que les gustaría visitar. Muchas respuestas coinciden en elegir Wano, especialmente "en su estado actual tras la derrota de Kaido". También aparecen otras opciones como Skypiea, Dressrosa, Egghead, Amazon Lily, Fishman Island o Water 7. Algunos comentarios:

- *"Definitely Wano based on the current day story. Now, Kaido is gone, seems like a cool place now with peace and order restored"*
- *"Wano and Skypiea post war and Water 7"*

<!--  -->

**Análisis del resumen generado por mT5**

En cuanto al resumen abstractivo generado por el modelo mT5, podemos ver que el resultado ha sido el siguiente:

- *"The BBC’s weekly The Boss series profiles different readers from around the world. This is a selection of places you choose to visit."*

Este resumen es claramente incorrecto. Aunque menciona de forma muy general la idea de “lugares que elegirías visitar”, introduce una referencia completamente distinta al hilo, concretamente a una serie de la BBC, que no aparece en ningún momento en la submission ni en los comentarios. Por tanto, el modelo no ha entendido correctamente el contexto de One Piece y ha generado una frase erróea con la que parece que está comentando algún otro tipo de submission.

<!--  -->

**Análisis del resumen generado por Gemma**

En cuanto al resumen abstractivo generado por Gemma, podemos ver que ha sido el siguiente:

- *"Este resumen captura la esencia del texto, enfocándose en la pregunta central sobre la elección de la isla para un personaje que se despierta en el mundo de One Piece."*

En este caso, el modelo identifica correctamente el tema general del hilo, ya que reconoce que se trata de una pregunta sobre qué isla elegir en caso de despertar dentro del mundo de One Piece. Sin embargo, el resultado no es exactamente un resumen.

El principal problema es que Gemma no ha resumido el contenido del hilo, sino que ha generado una descripción sobre como se ve el resumen. Es decir, produce una respuesta donde explica como serái el resumen, en lugar de resumir directamente la información obtenida de los textos. No menciona ninguna de las ideas principales de los comentarios, como las  islas que dicen los usuarios.

Por tanto, aunque el modelo demuestra que ha comprendido el contexto general, no llega a hacer un resumen adecuado.

<!--  -->

**Comparación entre ambos modelos**


Comparando ambos resultados, podemos ver que en este caso Gemma también ofrece un resultado superior al de mT5. Mientras que mT5 genera un resumen bastante incorrecto e introduce información ajena al hilo, Gemma sí identifica correctamente el tema principal de la discusión.

Sin embargo, aunque Gemma entiende el contexto, su respuesta no es un resumen real, sino una descripción sobre lo que debería ser un resumen.

En conclusión, aunque ninguno de los dos modelos produce un resumen adecuado, Gemma vuelve a mostrar un mejor rendimiento que mT5.

<!--  -->

#### **CUARTO HILO:**

**Descripción del hilo**

El cuarto hilo analizado también pertenece al subreddit de One Piece y tiene el id "1izidj7".

Su contexto se basa en la pregunta principal del título del hilo, donde se plantea si Sanji, en el momento actual del manga, es más fuerte que Zeff:

- Título: *"Is Sanji already stronger than Zeff?"*

El hilo no cuenta con una descripción adicional, por lo que el contexto se obtiene a partir del título y de los comentarios. En general, los usuarios debaten sobre cuál de los dos personajes es más fuerte, teniendo en cuenta tanto el estado actual de Sanji como el pasado de Zeff. La mayoría de comentarios dicen que Sanji lleva siendo más fuerte que Zeff durante bastante tiempo. Algunos comentarios:

- *"Definitely Wano based on the current day story. Now, Kaido is gone, seems like a cool place now with peace and order restored"*
- *"Without a question. Sanji surpassed him a while ago"*

<!--  -->

**Análisis del resumen generado por mT5**

En cuanto al resumen abstractivo generado por el modelo mT5, podemos ver que el resultado ha sido el siguiente:

- *"Luffy has become one of the most successful pirates in the new series, according to the BBC's weekly The Boss series."*

Este resumen no guarda ninguna relación con el contenido del hilo. Introduce información externa como el BBC y habla de Luffy, quien no es el personaje en el que se centra principalmente el hilo, lo que indica un fallo grave en el resumen. El modelo no solo no resume el contenido, sino que genera una frase ajena al contexto.

<!--  -->

**Análisis del resumen generado por Gemma**

En cuanto al resumen abstractivo generado por Gemma, podemos ver que ha sido el siguiente:

- *"Sanji surpassed Zeff significantly when he was still at his peak. He was stronger than Zeff during the last few chapters of the series. The fact that he was known for his immense power within the New World, meant that he could have"*

En este caso, el modelo identifica correctamente el tema general del hilo, ya que reconoce que se trata de una pregunta sobre qué personaje es más fuerte si Sanji o Zeff y recoge algunas ideas de los comentarios donde dice que que Sanji ha superado a Zeff.

Sin embargo, el resumen presenta un problema importante. El texto generado comienza siendo un buen resumen pero al final se corta de repente el texto con la frase “meant that he could have”, dejando la idea incompleta y mostrando un fallo en la generación del texto.

Aun así, aunque el resultado no sea perfecto, el resumen tiene relación con el contexto del hilo y refleja correctamente la temática, aal contrario que mt5, por lo que Gemma vuelve a ser mejor que mt5.

<!--  -->

#### **QUINTO HILO:**

**Descripción del hilo**

El quinto hilo analizado pertenece esta vez, al subreddit de **drawing** y tiene el id "1huadwj".

Su contexto se basa en la pregunta principal del título del hilo, donde una persona cuestiona si debería vender uno de sus sketchbooks después de recibir una oferta de 800 dólares.

- Título: *"I was offered $800 to sell this sketchbook. is it worth giving it? help pls"*

El hilo no incluye una descripción, por lo que toda la discusión se produce a través de los comentarios de los usuarios. En esta caso, observamos opiniones bastante divididas. En general, muchas de personas opinan que el sketchbook tiene un gran valor personal y artístico, por lo que muchos recomiendan que no lo venda. En cambio, hay otras personas que le dicen que sí que lo venda, pues lo va a tener "cogiendo polvo". Además, hay algunas personas que sospechan un poco sobre la oferta, pensando que podría tratarse de una estafa o de alguien interesado en usarlo con fines comerciales.

- *"Don’t sell that bruh that’s priceless"*
- *"Money is good, but memories are better. Keep it..."*

- *"that's strange"*
- *"Better than it collecting dust in shelve."*

- *"If you do sell it, and they are gonna use it to profit (resell your art)..."*

<!--  -->

**Análisis del resumen generado por mT5**

En cuanto al resumen abstractivo generado por el modelo mT5, podemos ver que el resultado ha sido el siguiente:

- *"This is a sketchbook written by an artist who has been offered $800 to sell. The drawings of the artist have been revealed by the BBC's weekly The Boss."*

Este resumen recoge adecuadamente la idea del hilo, diciendo que es un sketchbook de un artista al que le han ofrecido 800 dólares si lo vende. Sin embargo, vuelve a introducir información completamente ajena al contenido original, mencionando de nuevo a la BBC (por algún motivo siempre nombra la BBC).

Por tanto, aunque el modelo consigue identificar el tema, el resultado sigue incluyendo información incorrecta, haciendo que sea el resumen sea poco fiable. 

<!--  -->

**Análisis del resumen generado por Gemma**

En cuanto al resumen abstractivo generado por Gemma, podemos ver que ha sido el siguiente:

- *"The text describes a situation where an artist is offering a significant sum (around $800) for a sketchbook, seemingly as a way to acquire a \"legacy\" for their own artistic endeavors. The artist is concerned about the value of the sketchbook"*

En este caso, Gemma identifica parcialmente el tema del hilo, ya que aunque que habla sobre el sketchbook y la cantidad de 800 dólares, interpreta mal la situación. El resumen da a entender que es el artista el que está ofreciendo dinero para comprar un sketchbook, cuando en realidad es a él a quién quieren comprarle el sketchbook.

Además, nos habla sobre  sobre adquirir un “legacy” artístico, algo que no aparece directamente en los comentarios. Si es cierto que un comentario nombra algo similar, diciendole que haga un contrato con un abogado antes de venderlo por si acaso:

- *"Careful what you do. Go to a lawyer get a contract sell it to him for 800 with fine print you never know when someone's going to blow up with your ideas"*

Por tanto, aunque Gemma se acerca más al tema que mT5, el resultado presenta errores  de interpretación y no resume de forma completamente correcta.


<!--  -->

#### **SEXTO HILO:**

**Descripción del hilo**

El sexto hilo analizado pertenece esta vez, al subreddit de **leagueoflegends** y tiene el id "1kohidt".

Su contexto se basa en la pregunta del título del hilo, donde el usuario dice que en el juego no hay un personaje tipo "bruiser melee" (personaje cuerpo a cuerpo que combine daño y tanque a partes iguales) que utilice espada y escudo al mismo tiempo. El hilo incluye además una descripción donde dice que le parece una idea muy obvia e interesante y  aclara que Leona no cuenta porque es un "support" (personaje de apoyo) tanque y no un "bruiser".

- Título: *"How come there's no melee bruiser with sword and shield???"*
- Descripción: *"Just thought about it. Such an obvious and cool idea, and it hasn't been created yet. Leona doesn't count, she's a tank support."*

En general, los usuarios discuten sobre por qué nunca ha añadido un campeón así, diciendo personajes que podrían acercarse a esa idea, como Pantheon o Braum, además de proponer posibles habilidades y mecánicas en caso de que el campeón existiese. Algunos usuarios dicen que el personaje sería muy interesante desde el punto de vista del diseño y la jugabilidad, pero hay otros que dicen que nunca vamos a tener un personaje así, comentando que "Pantheon" es lo más cercano que vamos a tener o que los escudos no atractivos visualmente para el diseño de un personaje.

- *"I think it's just because there's barely any champs with shields. Isn't it just Leona, Pantheon, and Braum? I think shields are just generally not that exciting an aspect of design"*

- *"So like a champ that their Q has three charges. Then their W can be a shield bash stunn. Then their E can just be them raising their shield to absorb damage. Lastly their ultimate should just be a big steroid that also has an execute when recast on it fr their sword. That type of champ would have a lot of skill expression. Maybe a region like Noxus is the best place I see the champs theme."*

- *"Pantheon is the closest we will ever get to that."*

<!--  -->

**Análisis del resumen generado por mT5**

En cuanto al resumen abstractivo generado por el modelo mT5, podemos ver que el resultado ha sido el siguiente:

- *"It's been a huge success for the first time since the release of the Battle of Thrones ."*

Una ves más, el modelo mT5 no ha hecho un buen trabajo a la hora de resumir el texto, generando una respuesta completamente ajena a lo que trata el hilo. Por lo tanto, el resumen generado es malo.

<!--  -->

**Análisis del resumen generado por Gemma**

En cuanto al resumen abstractivo generado por Gemma, podemos ver que ha sido el siguiente:

- *"Here's a summary of the text:  '**"How come there's no melee bruiser with sword and shield?"**'. The text describes a debate about the lack of a traditional \"melee\" character in the game, specifically the absence"*

En este caso, Gemma parece identificar un poco que el contexto de la pregunta del hilo escribiendola textualmente en el resumen. Pero cuando comienza a describir el texto sí que habla de la ausencia de un personaje, pero no habla sobre lo más importante: que es un "bruiser melee" o sobre la espada y es escudo.  

Además, el resumen incluye texto innecesario como "Here's a summary of the text:", algo que no aporta información relevante al resumen. Además, se queda incompleto al final ya que la frase termina de repente en “specifically the absence…”, dando la impresión de que la respuesta se corta antes de tiempo.

<!--  -->

**Comparación entre ambos modelos**


Comparando ambos resultados, podemos ver que en este caso Gemma también ofrece un resultado superior al de mT5, aunque no muy superior. Mientras que mT5 genera un resumen totalmente incorrecto, Gemma identifica más o menos el tema y nombra las cosas más relevantes del hilo.

En conclusión, aunque ninguno de los dos modelos produce un resumen adecuado del todo, Gemma vuelve a mostrar un mejor rendimiento..

<!--  -->

#### **SÉPTIMO HILO:**

**Descripción del hilo**

El séptimo hilo analizado pertenece esta vez, al subreddit de **movies** y tiene el id "1ia6qt0".

Su contexto se basa en la pregunta planteada por el autor del hilo, donde rpegunta qué final de película te ha hecho llorar más. Además incluye una descripción en la que habla sobre las dos películas que más le han hecho a llorar a él.

- Título: *"What movie ending made you cry the most?"*
- Descripción: *"For me, there are two: The Return of the King and Kong: Skull Island. I watched both movies in the last 4 days, and ...*

En general, los usuarios responden con películas con finales que les hicieron llorar o les provocaron un fuerte impacto emocional. Algunas de las películas mencionadas son títulos como Your Lie in April o Terminator 2. Muchos comentarios incluyen además pequeñas explicaciones sobre por qué esos finales les afectaron más emocionalmente.

- *"Your Lie in April gotta be up there. Yes it’s an anime but there’s a movie, too. They cut most of the filler ..."*
- *"Terminator 2. 👍"*
- *"Looking for a friend for the end of the world. The last scene hit me hard."*

<!--  -->

**Análisis del resumen generado por mT5**

En cuanto al resumen abstractivo generado por el modelo mT5, podemos ver que el resultado ha sido el siguiente:

- *"The return of the King and Kong: Skull Island is a hugely emotional ending, according to the film's creator Nicholas Barber."*

Este resumen consigue captar parcialmente el tema principal del hilo, ya que menciona correctamente "The Return of the King" y "Kong: Skull Island", que son películas que aparecen en la descripción. Además, también identifica que el hilo va sobre finales emotivos.

Sin embargo, el resumen presenta varios errores. Menciona a Nicholas Barber como supuesto creador, cuando este nombre no aparece en ningún momento en el hilo, por lo que el modelo se está inventando información. Además, el resumen se centra únicamente en las películas mencionadas por el autor y no refleja realmente que el hilo es una discusión general de finales de películas que hacen llorar.

Por ello, aunque mT5 consigue captar parcilamente el tema del hilo, el resultado sigue es poco preciso y contiene información incorrecta.

<!--  -->

**Análisis del resumen generado por Gemma**

En cuanto al resumen abstractivo generado por Gemma, podemos ver que ha sido el siguiente:

- *"Here's a summary of the text: The text describes a deeply emotional response to a film, specifically the ending of \"Vanilla Sky\" and \"The Color Purple.\" The narrator expresses a strong emotional reaction to the ending, characterized by crying and"*

En este caso, Gemma identifica correctamente que el hilo trata sobre reacciones emocionales a finales de películas y menciona algunas películas que aparecen en los comentarios del hilo, como "Vanilla Sky" y "The Color Purple".

Sin embargo, el resumen también presenta varios problemas. En primer lugar, incluye texto innecesario como "Here's a summary of the text", algo que no aporta contenido relevante, por otro lado el resumen queda incompleto, ya que termina de repente en "characterized by crying and", indicando que la generación se corta antes de tiempo.

<!--  -->

**Comparación entre ambos modelos**


Comparando ambos resultados, podemos ver que en este caso tanto Gemma como mT5 consigue resumenes similares, nombrando ambos películas mencionadas y hablando de finales triste y emotivos. Además, ambos presentan algunos errores que hacen que ninguno tenga un resumen perfecto.

Por lo tanto, en este caso se podría decir que no hay un mejor modelo ya que los dos hacen un trabajo parecido.

<!--  -->

#### **OCTAVO HILO:**

**Descripción de hilo**

El octavo hilo analizado pertenece esta vez, al subreddit de **soccer** y tiene el id "1otomoq".

Su contexto se basa en una noticia sobre Lionel Messi y la vez que hizo una visita al Camp Nou de imprevisto, donde estaba sólo en el estadio. El título dice que Messi no contactó con el Barça para acceder al Camp Noy. En la descripción del hilo se explica que según fuentes cercanas a Messi, entró en el Camp Nou un domingo a medianoche sin avisar a nadie ni pedir permiso por una una decisión espontánea mientras estaba en Barcelona.

- Título: *"Messi did not contact Barça to gain access to Camp Nou"*

- Descripción: *"According to sources close to Messi, the former Barça number 10 entered the stadium facilities on Sunday, around midnight, without contacting anyone at the club to ask for permission or give notice. It was a spur-of-the-moment decision by the Argentine star who, while in Barcelona..."*

En los comentarios, la mayoria de usuarios reaccionan con humor o normalizan la situación, diciendo que Messi no necesita permiso para entrar en “su casa” o que en Barcelona puede hacer lo que quiera. 

- *"Messi can do whatever he pleases in Barcelona as I'm concerned."*
- *"You don’t need permission to get into your home"*

<!--  -->

**Análisis del resumen generado por mT5**

En cuanto al resumen abstractivo generado por el modelo mT5, podemos ver que el resultado ha sido el siguiente:

- *"FC Barcelona has confirmed that Lionel Messi entered the Camp Nou stadium without permission."*

En este caso, el resumen generado por mT5 capta el contenido principal del hilo, ya que menciona a Messi, el Camp Nou y la idea de que entró sin permiso. Sin embargo, el resumen no es del todo preciso, porque dice que el FC Barcelona confirmó que Messi entró sin permiso, cuando en el texto original se dice lo contrario. En la descripción se dice que el club dice que sí hubo una autorización del personal de seguridad.

- *"The official version offered by the club to the media almost two hours after Messi's post was that Leo went to the stadium and requested permission to access the Spotify Camp Nou from Limak security personnel..."*

Por tanto, mT5 consigue captar el tema general del texto, pero interpreta mal un detalle importante. 

<!--  -->

**Análisis del resumen generado por Gemma**

En cuanto al resumen abstractivo generado por Gemma, podemos ver que ha sido el siguiente:

- *"Messi bypassed security and entered Camp Nou on Sunday, around midnight, without permission. This was a deliberate action to troll the club president, who was concerned about the public's reaction. The intention was to disrupt the club’s strategy and capture attention"*

En este caso, Gemma identifica correctamente algunos elementos importantes del hilo, como que Messi entró al Camp Nou un domingo en medianoche y que fue sin avisar ni contactar con el club.

Sin embargo, el resumen introduce información que no es del todo correcta como que Messi lo hizo para "trollear" al presidente del club y alterar la estrategia del Barça, pero este es solamente algunas ideas que tienen usuarios y que aparecen únicamente en algunos comentarios. Pero dentro de lo que cabe, al nombrarse varias veces, Gemma hace un buen trabajo en añadirlo al resumen pues su único contexto son los pocos comentarios que presenta el hilo y no la situación real.

- *"Interesting time to do it, when Barça team and top executives are in Vigo for their Laliga match. Messi did it intentionally to troll the club president..."*

<!--  -->

#### **NOVENO HILO:**

**Descripción de hilo**

El noveno hilo analizado pertenece esta vez, al subreddit de **gaming** y tiene el id "1lwcci5".

Su contexto se basa en la pregunta del título donde el autor pregunta a los usuarios que juegos les suelen recomendar constantemente, pero que todavía no han jugado. 

- Título: *"What are games you keep being recommended all the time but never played?"*

En la descripción, el autor menciona Outer Wilds, diciendo que se lo han recomendado muchas veces pero que no lo ha jugado y que debería probarlo algún día. También menciona otros videojuegos como Prey, Far Cry 3 y Mass Effect.

- Descripción: *"Outer fucking Wilds. If I had a nickel for every time I was suggested to play this game, I'd have... many nickels. I have to check it out at this point... someday. Also there is Prey, Far Cry 3 and Mass Effect trilogy."*

En los comentarios, la mayoría de usuarios responden con juegos que pese a haber recibido muchas recomendaciones, no han jugado nunca como Dark Souls, Elden Ring o Yakuza. Además, varios comentarios hablan sobre Outer Wilds, diciendole al autor que es un juego estupendo o que les pareció un poco aburrido. Se repite mucho que el juego se confunde con Outer Worlds.

- *"Elden Ring. My friends keep recommending me this game but..."*
- *"I would LOVE to forget all the important stuff from OW"*
- *"I finally broke down and played Outer Wilds and found it really boring."*
- *"I accidentally bought outer worlds thinking it was wilds..."*

<!--  -->

**Análisis del resumen generado por mT5**

En cuanto al resumen abstractivo generado por el modelo mT5, podemos ver que el resultado ha sido el siguiente:

- *"Outer Wilds has been named the best video game of all time, according to the creator Gta Strongly."*

Este resumen detecta el elemento principal del hilo, ya que menciona el videojuego Outer Wilds. También dice que Outer Wilds ha sido nombrado el mejor videojuego de todos los tiempos porque se comenta en un comentario y atribuye esa afirmación a “the creator Gta Strongly”, algo que no se dice en el hilo.

- *"Outer Wilds holds my award for greatest video game of all time...*

Además, el resumen se centra únicamente en Outer Wilds, ignorando que la pregunta del hilo era sobre todos los juegos que los usuarios reciben como recomendación, pero que aún no han jugado. Por tanto, aunque mT5 identifica una palabra clave, no resume correctamente el texto.

<!--  -->

**Análisis del resumen generado por Gemma**

En cuanto al resumen abstractivo generado por Gemma, podemos ver que ha sido el siguiente:

- *""Here’s a summary of the text: The author is deeply engrossed in the game “Outer Wilds.” They are obsessed with it, considering it a treasured possession, and are perpetually seeking out information about it. They are incredibly passionate about"*

En este caso, Gemma también identifica que Outer Wilds es un elemento importante del hilo, pero interpreta mal el contexto general del hilo. Igual que mT5, se centra en Outer Wilds y no en que la pregunta del hilo es sobre todos los juegos que los usuarios reciben como recomendación, pero que aún no han jugado. Además, el autor no dice que esté profundamente enganchado al juego ni que esté obsesionado con él, precisamente porque no lo ha jugado.

Por tanto, Gemma también falla al interpretar correctamente el contexto de la publicación.

<!--  -->

#### **DÉCIMO HILO:**

**Descripción de hilo**

El décimo hilo analizado pertenece también al subreddit de **gaming** y tiene el id "1mw43np".

Su contexto se basa en una noticia sobre el rendimiento de Elden Ring: Tarnished Edition en Nintendo Switch 2. El título dice que el juego funciona tan mal en esta consola que Bandai Namco no pemite que la gente lo grabe en un evento de videojuegos donde se presentó (Gamescom).

- Título: *"Elden Ring: Tarnished Edition Runs So Badly On Switch 2 That Bandai Namco Isn't Allowing Recording At Gamescom"*

El hilo no tiene descripción, por lo que el contexto se obtiene del título y de los comentarios. En general, los comentarios hablan de porque ocurre este mal rendimiento: si se debe a limitaciones de la Switch 2, a una mala optimización del juego, etc.. Algunos comparan el rendimiento con otras plataformas como PC o PlayStation. Pero en general, las mayoría de comentarios dice que no es una sorpresa que un juego tan exigente como Elden Ring tenga mal rendimiento en Nintendo Switch 2.

- *"Someone here explained that it’s not an optimizable issue but a clash in tech.*
- *"Whaaaaat? Noooooo. The Switch 2 not being able to keep up with a full fledged console???? Who could have predicted???"*
- *"People need to understand that the switch 2 has a better upscaler"*
- *" 'Big Surprise'. Said no one"*
- *"Their PC ports are always trash, but it's honestly surprising to see a consol port so bad..."*

<!--  -->

**Análisis del resumen generado por mT5**

En cuanto al resumen abstractivo generado por el modelo mT5, podemos ver que el resultado ha sido el siguiente:

- *"Elden Ring: Tarnished edition has been released on the Nintendo Switch 2, according to the makers of the game. Here is a selection of some of its best-selling games."*

Una vez más, este resumen identifica parcialmente el tema del hilo, ya que menciona Elden Ring y Nintendo Switch 2, pero no recoge el tema principal del hilo, que es el mal rendimiento del juego en la consola.

Además, el resumen introduce información que no aparece en el hilo, como que se trata de una selección de juegos más vendidos. 

Por tanto, aunque mT5 detecta algunos elementos clave, no logra resumir adecuadamente el contexto del hilo.

<!--  -->

**Análisis del resumen generado por Gemma**

En cuanto al resumen abstractivo generado por Gemma, podemos ver que ha sido el siguiente:

- *"Elden Ring runs exceptionally well on the Switch 2, with the game rendering in ultrawide resolutions, but the console has a 16:9 portion. The game places black bars to display it."*

En este caso, Gemma comete un error importante de interpretación. El hilo trata precisamente de que Elden Ring funciona muy mal en Switch 2, pero el resumen dice lo contrario, diciendo que el juego funciona "excepcionalmente bien en Switch 2". 

Además, el modelo se centra demasiado en un comentario concreto que habla del renderizado  y las barras negras.

- *Elden Ring actually renders in ultrawide resolutions (eating performance) but the game places black bars over..."*

Por tanto, aunque Gemma nombra ciertas ideas claves del hilo, esta vez genera un resumen, a mi parecer, peor que el de mT5, pues resumen la información diciendo lo contrario a lo que realmente se habla en el hilo.